# Alchemy GeomML + TDA — Финальные графики

Три эксперимента:
1. **bs=256** — baselines: FCNN, SchNet, EGNN
2. **bs=512** — только EGNN (для сравнения batch size)
3. **bs=1024** — все EGNN-варианты: EGNN, EGNN+TDA, EGNN Vector, EGNN Vector+TDA, EGNN Tensor, EGNN Tensor+TDA

**Запуск:**
- Kaggle/Colab: запустите ячейку 'Kaggle/Colab'
- Локально: запустите ячейку 'Локально'

In [ ]:
# === Kaggle/Colab ===
import os
if not os.path.exists('alchemy-geom-tda'):
    !git clone https://github.com/ThomasMoore25/alchemy-geom-tda.git
%cd alchemy-geom-tda
!git pull

In [ ]:
# === Локально (если репозиторий уже клонирован) ===
# Раскомментируйте и укажите путь к репозиторию:
# import os
# os.chdir('/path/to/alchemy-geom-tda')

In [ ]:
import glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt

FIG = 'results/figures/final'
os.makedirs(FIG, exist_ok=True)

def load_csv(path):
    return pd.read_csv(path)

# Эксперимент 1: bs=256
e1 = {}
for f in glob.glob('results/experiments/batch_size_256/history_*.csv'):
    name = os.path.basename(f).replace('history_', '').replace('_all.csv', '').replace('.csv', '')
    e1[name] = load_csv(f)
    print(f'E1 bs=256 {name}: {len(e1[name])} epochs')

# Эксперимент 2: bs=512 (EGNN only)
e2 = {}
for f in glob.glob('results/experiments/batch_size_512/history_egnn_all_*.csv'):
    e2['egnn'] = load_csv(f)
    print(f'E2 bs=512 egnn: {len(e2["egnn"])} epochs')

# Эксперимент 3: bs=1024 — загрузка history CSV
e3_hist = {}
for f in glob.glob('results/experiments/batch_size_1024/history_*.csv'):
    name = os.path.basename(f).replace('history_', '').rsplit('_', 2)[0]
    if name not in e3_hist:  # берём последний файл каждой модели
        e3_hist[name] = load_csv(f)
        print(f'E3 bs=1024 {name}: {len(e3_hist[name])} epochs')

# Эксперимент 3: bs=1024 — финальные метрики (из summary CSV и логов)
e3_results = {
    'EGNN':           {'mu': 0.284, 'alpha': 0.583, 'gap': 0.0058, 'loss': 0.344, 'epochs': 135, 'params': 951759, 'time_h': 3.5},
    'EGNN+TDA':       {'mu': 0.298, 'alpha': 0.619, 'gap': 0.0061, 'loss': 0.362, 'epochs': 146, 'params': 971831, 'time_h': 3.9},
    'EGNN Vector':    {'mu': 4.123, 'alpha': 0.354, 'gap': 0.0044, 'loss': 0.167, 'epochs': 188, 'params': 968023, 'time_h': 5.3},
    'EGNN Vec+TDA':   {'mu': 4.121, 'alpha': 0.510, 'gap': 0.0055, 'loss': 0.216, 'epochs': 108, 'params': 981439, 'time_h': 3.1},
    'EGNN Tensor':    {'mu': 4.102, 'alpha': 0.939, 'gap': 0.0087, 'loss': 0.366, 'epochs': 100, 'params': 941900, 'time_h': 2.9},
    'EGNN Ten+TDA':   {'mu': 4.111, 'alpha': 1.011, 'gap': 0.0104, 'loss': 0.428, 'epochs': 273, 'params': 948660, 'time_h': 8.1},
}
print(f'\nE3 bs=1024: {len(e3_results)} models')

# v26.1 test results
e1_test = {}
for name, df in e1.items():
    last = df.iloc[-1]
    e1_test[name.upper()] = {'mu': last['test_mu_mae'], 'alpha': last['test_alpha_mae'],
                             'gap': last['test_gap_mae'], 'loss': last['test_loss'], 'epochs': len(df)}
print(f'E1 test: {e1_test}')

## График 1: Baselines (bs=256) — Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
fig.suptitle('Эксперимент 1: Baselines (bs=256, полный датасет)', fontsize=16, fontweight='bold')

colors = {'FCNN': '#607D8B', 'SCHNET': '#795548', 'EGNN': '#2196F3'}
for ax, (col, title) in zip(axes.flat, [
    ('val_loss', 'Val Loss'), ('val_mu_mae', 'μ MAE (Debye)'),
    ('val_alpha_mae', 'α MAE (Bohr³)'), ('val_gap_mae', 'gap MAE (Hartree)')]):
    for name, df in e1.items():
        label = name.upper()
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=colors[label], label=label, linewidth=2)
    ax.set_title(title, fontsize=14); ax.set_xlabel('Epoch', fontsize=12)
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3)

plt.savefig(f'{FIG}/01_baselines_training.png', dpi=150, facecolor='white')
plt.show()

## График 2: Baselines — Test Metrics

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
fig.suptitle('Эксперимент 1: Test Metrics — FCNN vs SchNet vs EGNN (bs=256)', fontsize=14, fontweight='bold')
models = list(e1_test.keys()); colors = ['#607D8B', '#795548', '#2196F3']
for ax, (m, title, unit) in zip(axes, [('mu','μ MAE','D'), ('alpha','α MAE','Bohr³'), ('gap','gap MAE','Ha'), ('loss','Test Loss','norm')]):
    vals = [e1_test[mod][m] for mod in models]
    bars = ax.bar(models, vals, color=colors, edgecolor='black')
    ax.set_title(f'{title} ({unit})', fontsize=12); ax.grid(True, alpha=0.3, axis='y')
    bars[vals.index(min(vals))].set_edgecolor('gold'); bars[vals.index(min(vals))].set_linewidth(3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01, f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.savefig(f'{FIG}/02_baselines_test.png', dpi=150, facecolor='white')
plt.show()

## График 3: Batch Size Impact (EGNN)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
fig.suptitle('Эксперимент 2: Влияние Batch Size на EGNN', fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(e1['egnn']['epoch'], e1['egnn']['val_loss'], color='#2196F3', label='bs=256 (131 ep)', linewidth=2)
ax.plot(e2['egnn']['epoch'], e2['egnn']['val_loss'], color='#FF9800', label='bs=512 (164 ep)', linewidth=2)
ax.axhline(y=0.344, color='#4CAF50', linestyle='--', label='bs=1024 (135 ep, loss=0.344)')
ax.set_title('Val Loss', fontsize=13); ax.set_xlabel('Epoch'); ax.legend(fontsize=11); ax.grid(True, alpha=0.3)

ax = axes[1]
bs = ['bs=256', 'bs=512', 'bs=1024']; mu = [e1_test['EGNN']['mu'], 0.245, 0.284]; al = [e1_test['EGNN']['alpha'], 0.460, 0.583]; ga = [e1_test['EGNN']['gap'], 0.0046, 0.0058]
x = np.arange(3); w = 0.25
ax.bar(x-w, mu, w, label='μ MAE', color='#2196F3', edgecolor='black')
ax.bar(x, al, w, label='α MAE', color='#FF9800', edgecolor='black')
ax.bar(x+w, ga, w, label='gap MAE', color='#4CAF50', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(bs); ax.set_title('Test MAE', fontsize=13); ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis='y')

plt.savefig(f'{FIG}/03_batch_size.png', dpi=150, facecolor='white')
plt.show()

## График 4: Все EGNN-варианты (bs=1024) — Test Metrics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
fig.suptitle('Эксперимент 3: Все EGNN-варианты (bs=1024, 2 GPU T4)', fontsize=16, fontweight='bold')
models = list(e3_results.keys()); colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#E91E63']
for ax, (m, title, unit) in zip(axes.flat, [('mu','μ MAE','D'), ('alpha','α MAE','Bohr³'), ('gap','gap MAE','Ha'), ('loss','Test Loss','norm')]):
    vals = [e3_results[mod][m] for mod in models]
    bars = ax.barh(models, vals, color=colors, edgecolor='black')
    ax.set_title(title, fontsize=13); ax.grid(True, alpha=0.3, axis='x')
    bars[vals.index(min(vals))].set_edgecolor('gold'); bars[vals.index(min(vals))].set_linewidth(3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2, f'{val:.4f}', ha='left', va='center', fontsize=10, fontweight='bold')
plt.savefig(f'{FIG}/04_variants_test.png', dpi=150, facecolor='white')
plt.show()

## График 5: TDA Impact — с TDA vs без TDA (групповое сравнение)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 6), constrained_layout=True)
fig.suptitle('TDA Impact: с TDA vs без TDA (все модели, bs=1024)', fontsize=14, fontweight='bold')

pairs = [('EGNN', 'EGNN+TDA'), ('EGNN Vector', 'EGNN Vec+TDA'), ('EGNN Tensor', 'EGNN Ten+TDA')]
metrics = [('mu', 'μ MAE (D)'), ('alpha', 'α MAE (Bohr³)'), ('gap', 'gap MAE (Ha)'), ('loss', 'Test Loss')]
x = np.arange(len(pairs)); w = 0.35

for ax, (m_key, title) in zip(axes, metrics):
    no_tda = [e3_results[p[0]][m_key] for p in pairs]
    with_tda = [e3_results[p[1]][m_key] for p in pairs]
    ax.bar(x - w/2, no_tda, w, label='Без TDA', color='#2196F3', edgecolor='black')
    ax.bar(x + w/2, with_tda, w, label='С TDA', color='#FF9800', edgecolor='black')
    ax.set_xticks(x); ax.set_xticklabels(['EGNN', 'Vector', 'Tensor'], fontsize=11)
    ax.set_title(title, fontsize=12); ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis='y')
    for i, (n, t) in enumerate(zip(no_tda, with_tda)):
        delta = (t - n) / n * 100
        color = '#F44336' if delta > 0 else '#1a7f37'
        ax.text(i + w/2, t * 1.02, f'{"+" if delta > 0 else ""}{delta:.0f}%', ha='center', fontsize=9, color=color, fontweight='bold')

plt.savefig(f'{FIG}/05_tda_grouped.png', dpi=150, facecolor='white')
plt.show()

## График 6: EGNN vs EGNN Vector vs EGNN Tensor (объединённый)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 6), constrained_layout=True)
fig.suptitle('EGNN (скаляр) vs EGNN Vector (вектор μ) vs EGNN Tensor (вектор μ + тензор α)', fontsize=14, fontweight='bold')

compare = ['EGNN', 'EGNN Vector', 'EGNN Tensor']
colors_c = ['#2196F3', '#4CAF50', '#F44336']
x = np.arange(len(compare))

for ax, (m_key, title) in zip(axes, metrics):
    vals = [e3_results[mod][m_key] for mod in compare]
    bars = ax.bar(x, vals, color=colors_c, edgecolor='black', width=0.6)
    ax.set_xticks(x); ax.set_xticklabels(compare, fontsize=10, rotation=15)
    ax.set_title(title, fontsize=12); ax.grid(True, alpha=0.3, axis='y')
    bars[vals.index(min(vals))].set_edgecolor('gold'); bars[vals.index(min(vals))].set_linewidth(3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01, f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.savefig(f'{FIG}/06_scalar_vs_vector_vs_tensor.png', dpi=150, facecolor='white')
plt.show()

## График 7: EGNN Tensor — Training Curves (100 epochs)

In [ ]:
if 'egnn_tensor' in e3_hist:
    df = e3_hist['egnn_tensor']
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    fig.suptitle('EGNN Tensor Training (100 epochs, Часть B: вектор μ + тензор α)', fontsize=14, fontweight='bold')

    for ax, (col, title) in zip(axes, [('val_loss', 'Val Loss'), ('val_alpha_mae', 'α MAE (Bohr³)'), ('val_gap_mae', 'gap MAE (Ha)')]):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color='#F44336', linewidth=2)
            ax.axhline(y=df[col].min(), color='gray', linestyle='--', alpha=0.5, label=f'Best: {df[col].min():.4f}')
        ax.set_title(title, fontsize=12); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True, alpha=0.3)

    plt.savefig(f'{FIG}/07_tensor_training.png', dpi=150, facecolor='white')
    plt.show()
else:
    print('EGNN Tensor history not found')

## График 8: Параметры и время обучения

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
fig.suptitle('Параметры и время обучения (bs=1024)', fontsize=14, fontweight='bold')

models_short = ['EGNN', 'EGNN+TDA', 'EGNN\nVector', 'EGNN\nVec+TDA', 'EGNN\nTensor', 'EGNN\nTen+TDA']
params = [e3_results[m]['params'] for m in e3_results]
times = [e3_results[m]['time_h'] for m in e3_results]
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#E91E63']

ax = axes[0]
bars = ax.bar(models_short, params, color=colors, edgecolor='black')
ax.set_title('Параметры', fontsize=13); ax.set_ylabel('Count'); ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000, f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax = axes[1]
bars = ax.bar(models_short, times, color=colors, edgecolor='black')
ax.set_title('Время обучения (часы)', fontsize=13); ax.set_ylabel('Hours'); ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val:.1f}h', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.savefig(f'{FIG}/08_params_time.png', dpi=150, facecolor='white')
plt.show()

## График 9: TDA Feature Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
cats = ['H₀ Betti\n(16 bins)', 'H₁ Betti\n(16 bins)', 'H₂ Betti\n(16 bins)', 'Entropy+Mean\n(4 dims)']
total = [16, 16, 16, 4]; zero = [0, 12, 16, 0]; nonzero = [t-z for t,z in zip(total, zero)]
x = range(len(cats))
ax.bar(x, nonzero, color='#4CAF50', label='Информативные', edgecolor='black')
ax.bar(x, zero, bottom=nonzero, color='#F44336', label='Всегда ноль', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=12)
ax.set_ylabel('Количество фичей', fontsize=13)
ax.set_title('TDA Feature Quality: 54% всегда ноль\n(молекулы 9-29 атомов слишком простые)', fontsize=14, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3, axis='y')
ax.text(0.5, 0.95, '54% ALWAYS ZERO → BatchNorm → Шум → Ухудшение', transform=ax.transAxes, ha='center', fontsize=13, fontweight='bold', color='#F44336', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
plt.savefig(f'{FIG}/09_tda_features.png', dpi=150, facecolor='white')
plt.show()

## График 10: Сводная таблица

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8), constrained_layout=True)
ax.axis('off')

table_data = [
    ['Exp', 'Model', 'BS', 'Epochs', 'μ MAE', 'α MAE', 'gap MAE', 'Loss', 'Params'],
    ['1', 'FCNN', '256', '176', '0.851', '2.271', '0.026', '1.235', '53K'],
    ['1', 'SchNet', '256', '226', '0.131', '0.442', '0.003', '0.186', '320K'],
    ['1', 'EGNN', '256', '131', '0.179', '0.393', '0.004', '0.227', '952K'],
    ['2', 'EGNN', '512', '164', '0.245', '0.460', '0.005', '0.285', '952K'],
    ['3', 'EGNN', '1024', '135', '0.284', '0.583', '0.006', '0.344', '952K'],
    ['3', 'EGNN+TDA', '1024', '146', '0.298', '0.619', '0.006', '0.362', '972K'],
    ['3', 'EGNN Vector', '1024', '188', '4.123', '0.354', '0.004', '0.167', '968K'],
    ['3', 'EGNN Vec+TDA', '1024', '108', '4.121', '0.510', '0.006', '0.216', '981K'],
    ['3', 'EGNN Tensor', '1024', '100*', '4.102', '0.939', '0.009', '0.366', '942K'],
    ['3', 'EGNN Ten+TDA', '1024', '273', '4.111', '1.011', '0.010', '0.428', '949K'],
]
table = ax.table(cellText=table_data[1:], colLabels=table_data[0], cellLoc='center', loc='center')
table.auto_set_font_size(False); table.set_fontsize(12); table.scale(1.0, 1.8)
for j in range(9):
    table[0, j].set_facecolor('#37474F'); table[0, j].set_text_props(color='white', fontweight='bold')
colors_row = ['#ECEFF1', '#F5F5DC', '#E3F2FD', '#FFF3E0', '#E8F5E9', '#FFF3E0', '#F3E5F5', '#FFF3E0', '#FFEBEE', '#FCE4EC']
for i, color in enumerate(colors_row):
    for j in range(9):
        table[i + 1, j].set_facecolor(color)
for col, row in [(4, 1), (5, 2), (6, 1), (7, 7)]:
    table[row, col].set_text_props(fontweight='bold', color='#2E7D32')
ax.set_title('Все результаты: 3 эксперимента, 8 моделей', fontsize=14, fontweight='bold', pad=20)
ax.text(0.5, -0.03, '* EGNN Tensor: 100 эпох (Kaggle 12h лимит). EarlyStopping не сработал — обучение остановлено по лимиту времени.', transform=ax.transAxes, ha='center', fontsize=10, style='italic', color='gray')
plt.savefig(f'{FIG}/10_summary_table.png', dpi=150, facecolor='white', bbox_inches='tight')
plt.show()